In [20]:
# 1. Install PySpark & Initialize SparkSession
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, lower, to_date, when, count
import os
import shutil
from google.colab import files

spark = SparkSession.builder \
    .appName("Data Lake Preprocessing") \
    .getOrCreate()

# 2. Create Data Lake Folders FIRST
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)

# 3. Upload sales.csv into Bronze Layer
uploaded = files.upload()
shutil.move("sales.csv", "data_lake/bronze/sales.csv")

# 4. Load & Inspect Bronze Data
bronze_path = "data_lake/bronze/sales.csv"
raw_df = spark.read.csv(bronze_path, header=True, inferSchema=True)

raw_df.printSchema()
raw_df.show(5)
initial_count = raw_df.count()
print(f"Initial Row Count: {initial_count}")

# Check duplicates and missing values
duplicate_count = initial_count - raw_df.dropDuplicates().count()
print(f"Exact Duplicate Rows: {duplicate_count}")

raw_df.select([count(when(col(c).isNull(), c)).alias(c) for c in raw_df.columns]).show()

# 5. Preprocessing - Silver Layer
# Remove exact duplicates
processed_df = raw_df.dropDuplicates()

# Trim & Standardize Text Columns
text_cols = [f.name for f in processed_df.schema.fields if str(f.dataType) == "StringType"]
for col_name in text_cols:
    processed_df = processed_df.withColumn(col_name, trim(col(col_name)))
    processed_df = processed_df.withColumn(col_name, initcap(col(col_name)))

# Numerical Validations
if "quantity" in processed_df.columns:
    processed_df = processed_df.filter(col("quantity") > 0)
if "unit_price" in processed_df.columns:
    processed_df = processed_df.filter(col("unit_price") >= 0)
if "discount_percentage" in processed_df.columns:
    processed_df = processed_df.filter((col("discount_percentage") >= 0) & (col("discount_percentage") <= 100))

# Date Standardization
if "date" in processed_df.columns:
    processed_df = processed_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
                               .filter(col("date").isNotNull())

# Drop rows where all elements are missing
processed_df = processed_df.dropna(how="all")

# 6. Save to Silver Layer & Verify
silver_path = "data_lake/silver/preprocessed_sales"
processed_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(silver_path)

final_count = processed_df.count()
print(f"Row count before preprocessing: {initial_count}")
print(f"Row count after preprocessing:  {final_count}")

# Read back Silver dataset to verify
silver_df = spark.read.csv(silver_path, header=True, inferSchema=True)
silver_df.show(5)

Saving sales.csv to sales.csv
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+------------